# Romania 2001 county download and reconciliation progress

Run **Cells 1 and 2** to see all county reporting units in the saved Ministry inventory. Then rerun **Cell 2 anytime**, including while acquisition is running. This notebook makes **no internet requests** and does not change download checkpoints or run outcome analyses.

The overview separates school applicant pages, school results (admitted and rejected), incoming-student lists, and saved reconciliation checks. **A downloaded page is not the same as a reconciled dataset.** An old incomplete county list can coexist with later school pages that fill its gaps; see the detailed audit findings before deciding more downloads are necessary.

The source inventory uses its historical county reporting units: for example, B is **București and Ilfov** in these sources. It is not a new list of present-day administrative boundaries.


In [ ]:
# CELL 1 — Visible settings
from pathlib import Path

# Where the acquisition workers save their private checkpoints.
PRIVATE_ROOT = Path.home() / "Desktop" / "VECTOR_temp"

# [] shows every county. To focus temporarily, use e.g. ["B", "AB", "TM"].
COUNTIES_TO_SHOW = []

# The county whose individual download families and audit findings appear in Cell 3.
DETAIL_COUNTY = "B"

# False keeps this entirely read-only. True writes new aggregate CSV snapshots only.
SAVE_SNAPSHOT = False

# Locate education_analysis from the notebook folder or the repository root.
# If you move the notebook elsewhere, set EDUCATION_ROOT directly to that folder.
EDUCATION_ROOT = None
for parent in [Path.cwd(), *Path.cwd().parents]:
    for candidate in [parent, parent / "education_analysis",
                      parent / "3-Master_Plan/VECTOR_work/new_VECTOR_work/education_analysis"]:
        if (candidate / "code/EDUCATION_20261013_romania_county_progress.py").exists():
            EDUCATION_ROOT = candidate
            break
    if EDUCATION_ROOT is not None:
        break
if EDUCATION_ROOT is None:
    raise FileNotFoundError("Set EDUCATION_ROOT to the education_analysis folder containing code/ and notebooks/.")
SNAPSHOT_FOLDER = EDUCATION_ROOT / "outputs" / "romania_county_progress_snapshots"


In [ ]:
# CELL 2 — Refresh the all-county table. Rerun this cell whenever you want an update.
import importlib
import sys

if str(EDUCATION_ROOT / "code") not in sys.path:
    sys.path.insert(0, str(EDUCATION_ROOT / "code"))
import EDUCATION_20261013_romania_county_progress as progress
importlib.reload(progress)

print("Reading saved county progress and audit records; no requests to Wayback...", flush=True)
county_progress = progress.collect_progress(PRIVATE_ROOT, EDUCATION_ROOT / "outputs")
all_counties = county_progress["counties"]  # Full machine-readable table for your own sorting.
visible_counties = progress.show_progress(county_progress, COUNTIES_TO_SHOW)
if SAVE_SNAPSHOT:
    for path in progress.save_snapshot(county_progress, SNAPSHOT_FOLDER):
        print("Saved aggregate snapshot:", path)


In [ ]:
# CELL 3 — Explain one county. Change DETAIL_COUNTY in Cell 1, or here, and rerun.
progress.show_details(county_progress, DETAIL_COUNTY)


## What the numbers do and do not establish

- **School applicant pages:** saved files from the earlier school-level score acquisition, compared with the original directory’s school count. File presence is counted here; hashes and student identities are not rechecked on each refresh.
- **School result pages:** saved files containing admitted and rejected outcomes, plus directory pages. The denominator is the larger of known links and the worker’s projection. A missing denominator displays `?` rather than inventing a total.
- **Incoming lists:** three distinct report types: candidates from other counties, those admitted, and those rejected. Each must cover its currently known/projected pages with no recorded unresolved URLs before it counts toward 3/3.
- **Original report families:** the initial six county-level sources. Their old manifest may remain incomplete even when later school or program reports recover the missing information.
- **Program recovery:** the last recorded verification of alternate program reports against occupancy. This is a saved reconciliation result, not a new verification performed by this notebook.
- **Reconciliation:** saved comparisons of applicants, school outcomes, destinations, and totals. Cell 3 shows the specific counts and the source CSV for each check. A matched printed name and score is corroboration, not proof of a unique national identity.
- **Dates:** audit dates are file modification times, not guaranteed dates of the scientific computation. “Downloads newer than report” is a conservative reminder to review freshness, not proof that the result has changed. Copying files can also change modification times.
- **Storage:** the detailed county summary reports selected source checkpoint files in decimal MB. It excludes logs, backups, shared metadata, and other caches, so it will not equal the size of your entire download folder. Repeated views can contain the same student; page or row counts are never a unique-student total.

This is a snapshot while other notebooks may be writing. If a count changes during refresh or a file cannot be read, rerun Cell 2. No county is automatically declared ready for analysis, and no missing record is written off as unavailable.
